# 05 — Tập mẫu chung và chia theo thời gian

Dự báo log return phiên kế tiếp, sau đó suy ra giá và hướng. Chạy các ô từ trên xuống. Dữ liệu dừng tại 30/09/2025.

## Chuẩn bị môi trường

Trên máy cá nhân, cài `pip install -r requirements.txt` trước. Các lệnh mặc định dùng snapshot được lưu trong dự án; không tự thay dữ liệu bằng lần tải mạng mới.

In [ ]:
# Tìm thư mục gốc khi chạy từ repo, từ notebooks/ hoặc trên Google Colab.
from pathlib import Path
import sys, subprocess, json, os
candidates = [Path.cwd(), Path.cwd().parent, Path('/content/fpt-stock-trends-prediction')]
ROOT = next((p for p in candidates if (p/'src/train.py').exists()), None)
if ROOT is None and 'google.colab' in sys.modules:
    subprocess.run(['git','clone','https://github.com/thiemhoang4807/fpt-stock-trends-prediction.git','/content/fpt-stock-trends-prediction'], check=True)
    ROOT = Path('/content/fpt-stock-trends-prediction')
if ROOT is None:
    raise FileNotFoundError('Hãy mở notebook trong repository đã tải về.')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT/'src'))
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd
from IPython.display import display, Image
print('Thư mục làm việc:', ROOT)


## Cùng ngày để so sánh công bằng

4.671 hàng OHLCV lưu gốc còn 4.669 phiên sau bỏ hai ngày HOSE đóng cửa. 49 hàng đầu dành cho chỉ báo kỹ thuật, còn 4.620 hàng. Sau yêu cầu 60 phiên đủ 42 biến, có 3.032 hàng chung, trong đó **3.031 hàng có nhãn**.

- Đầu vào đủ biến sớm nhất ngày 15/05/2013.
- Đầu vào có đủ cửa sổ ngày 06/08/2013, mục tiêu là 07/08/2013.
- Hàng cuối 30/09/2025 dùng để dự báo phiên kế tiếp.

In [ ]:
from prepare import prepare,build_experiment_table,feature_groups,DEFAULT_INPUT,DEFAULT_OUTPUT
prepare(DEFAULT_INPUT,DEFAULT_OUTPUT)
base,common,_=build_experiment_table(DEFAULT_INPUT)
print({name:len(cols) for name,cols in feature_groups().items()})

## Chia 85% Train, 15% Test

Có 2.576 nhãn Train từ 07/08/2013 đến 29/11/2023 và 455 nhãn Test từ 30/11/2023 đến 30/09/2025. Trong Train, kiểm định chéo gồm 5 lượt. Mỗi lượt mở rộng phần học về phía trước và chừa một hàng trước đoạn chấm.

In [ ]:
from train import _chronological_splits,cv_splits
data=common.dropna(subset=['next_log_return'])
masks,bounds=_chronological_splits(data)
print(json.dumps(bounds,indent=2))
display(pd.DataFrame([{'fold':i,'train_rows':len(a),'validation_rows':len(b),'gap_rows':b[0]-a[-1]-1} for i,(a,b) in enumerate(cv_splits(int(masks['train'].sum())),1)]))